# ✅ Notebook 06 — Model Validation
> **Market Demand Trend Analysis** | *Accuracy Assessment & Model Comparison*

**Objectives**
- Time-based train/test split (last 25% as hold-out)
- Evaluate all three models on test set: MAE, RMSE, MAPE
- Residual analysis: normality, autocorrelation
- Visualise actual vs. predicted for each model
- Produce a final model comparison leaderboard
- Recommend the best model for production use

---

## 0. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.stattools import durbin_watson
from prophet import Prophet

from utils import (
    set_style, load_merged, add_role_category, build_daily_series, train_test_split_ts,
    mae, rmse, mape, metrics_table,
    savefig, save_plotly, PALETTE, PLOTLY_TEMPLATE, OUT_REP
)

set_style()
print('✅ Setup complete')

## 1. Load Data & Build Series

In [ ]:
clean_path = OUT_REP / 'merged_clean.parquet'
if clean_path.exists():
    df = pd.read_parquet(clean_path)
else:
    df = load_merged()
    df = add_role_category(df)

daily = build_daily_series(df, date_col='first_seen')

train, test = train_test_split_ts(daily, test_frac=0.25)
HORIZON = len(test)

print(f'Train: {len(train)} days | Test: {len(test)} days | Horizon: {HORIZON}')
print('\nTest period:')
print(test)

## 2. Fit & Predict — All Models

In [ ]:
predictions = {}

# ── SARIMA ─────────────────────────────────────────────────────────────────
try:
    sarima = SARIMAX(train, order=(1, 1, 1),
                     seasonal_order=(1, 0, 1, 3),
                     enforce_stationarity=False,
                     enforce_invertibility=False).fit(disp=False)
    sarima_pred = sarima.get_forecast(steps=HORIZON).predicted_mean
    sarima_pred.index = test.index
    predictions['SARIMA'] = sarima_pred.clip(lower=0)
    print('✅ SARIMA fitted')
except Exception as e:
    print(f'❌ SARIMA failed: {e}')

# ── Prophet ────────────────────────────────────────────────────────────────
try:
    prop_df = train.reset_index()
    prop_df.columns = ['ds', 'y']
    prop_df['ds'] = pd.to_datetime(prop_df['ds'])

    pm = Prophet(daily_seasonality=True, weekly_seasonality=True,
                 yearly_seasonality=False, changepoint_prior_scale=0.3)
    pm.fit(prop_df)

    future = pm.make_future_dataframe(periods=HORIZON)
    prop_fc = pm.predict(future).tail(HORIZON)[['ds', 'yhat']]
    prop_fc = prop_fc.set_index('ds')['yhat'].clip(lower=0)
    prop_fc.index = test.index
    predictions['Prophet'] = prop_fc
    print('✅ Prophet fitted')
except Exception as e:
    print(f'❌ Prophet failed: {e}')

# ── ETS ────────────────────────────────────────────────────────────────────
try:
    ets = ExponentialSmoothing(
        train, trend='add',
        seasonal='add' if len(train) >= 6 else None,
        seasonal_periods=3 if len(train) >= 6 else None,
        initialization_method='estimated'
    ).fit(optimized=True)
    ets_pred = ets.forecast(HORIZON).clip(lower=0)
    ets_pred.index = test.index
    predictions['ETS'] = ets_pred
    print('✅ ETS fitted')
except Exception as e:
    print(f'❌ ETS failed: {e}')

## 3. Compute Metrics

In [ ]:
actual = test.values
all_metrics = []

for model_name, pred in predictions.items():
    m = metrics_table(actual, pred.values, model_name)
    all_metrics.append(m)

leaderboard = pd.concat(all_metrics, ignore_index=True)
leaderboard = leaderboard.sort_values('RMSE')
print('=== MODEL LEADERBOARD ===')
print(leaderboard.to_string(index=False))

In [ ]:
# Visual leaderboard
metrics_long = leaderboard.melt(id_vars='Model', value_vars=['MAE','RMSE','MAPE%'],
                                 var_name='Metric', value_name='Value')

fig = px.bar(
    metrics_long,
    x='Model', y='Value', color='Metric',
    barmode='group',
    title='Model Comparison — MAE / RMSE / MAPE',
    template=PLOTLY_TEMPLATE,
    color_discrete_map={
        'MAE'   : PALETTE['primary'],
        'RMSE'  : PALETTE['secondary'],
        'MAPE%' : PALETTE['danger'],
    }
)
fig.update_layout(height=420)
save_plotly(fig, '06_model_leaderboard')
fig.show()

leaderboard.to_csv(OUT_REP / '06_model_metrics.csv', index=False)
print('✅ Metrics saved → outputs/reports/06_model_metrics.csv')

## 4. Actual vs. Predicted — All Models

In [ ]:
fig = go.Figure()

# Full actual series
fig.add_trace(go.Scatter(
    x=daily.index, y=daily.values,
    name='Actual', mode='lines+markers',
    line=dict(color=PALETTE['primary'], width=2.5),
    marker=dict(size=7)
))

# Train/test boundary
fig.add_vline(x=str(train.index[-1]), line_dash='dash',
              line_color='#888', annotation_text='Test →')

# Model predictions
pred_colors = [PALETTE['secondary'], PALETTE['success'], PALETTE['info']]
for (name, pred), color in zip(predictions.items(), pred_colors):
    fig.add_trace(go.Scatter(
        x=pred.index, y=pred.values,
        name=f'{name} (test)',
        mode='lines+markers',
        line=dict(color=color, width=2, dash='dot'),
        marker=dict(size=6, symbol='diamond')
    ))

fig.update_layout(
    title='Actual vs Predicted — All Models on Hold-Out Test Set',
    xaxis_title='Date', yaxis_title='Daily Postings',
    template=PLOTLY_TEMPLATE, height=480,
    legend=dict(orientation='h', y=1.1, x=0)
)
save_plotly(fig, '06_actual_vs_predicted')
fig.show()

## 5. Residual Analysis

In [ ]:
n_models = len(predictions)
fig, axes = plt.subplots(n_models, 3, figsize=(16, 4 * n_models))
if n_models == 1:
    axes = [axes]

for row_idx, (model_name, pred) in enumerate(predictions.items()):
    residuals = actual - pred.values
    ax_res, ax_hist, ax_acf = axes[row_idx]

    # ── Residual over time ────────────────────────────────────────────────
    ax_res.plot(test.index, residuals,
                color=PALETTE['danger'], lw=1.8, marker='o', markersize=5)
    ax_res.axhline(0, color='#888', lw=1, linestyle='--')
    ax_res.fill_between(test.index, residuals, alpha=0.2, color=PALETTE['danger'])
    ax_res.set_title(f'{model_name} — Residuals', fontweight='bold')
    ax_res.set_ylabel('Error')
    ax_res.tick_params(axis='x', rotation=20)

    # ── Histogram + Normal fit ────────────────────────────────────────────
    ax_hist.hist(residuals, bins='auto', color=PALETTE['info'],
                 alpha=0.8, edgecolor='none', density=True)
    mu, sigma = residuals.mean(), residuals.std()
    x_norm = np.linspace(residuals.min(), residuals.max(), 200)
    ax_hist.plot(x_norm, stats.norm.pdf(x_norm, mu, sigma),
                 color=PALETTE['secondary'], lw=2, label='Normal fit')
    ax_hist.set_title(f'{model_name} — Residual Distribution', fontweight='bold')
    ax_hist.legend(fontsize=8)

    # ── ACF of residuals ──────────────────────────────────────────────────
    n_lags = max(1, min(3, len(residuals) // 2 - 1))
    plot_acf(residuals, lags=n_lags, ax=ax_acf,
             color=PALETTE['primary'],
             title=f'{model_name} — ACF of Residuals')
    ax_acf.set_facecolor('#252535')

plt.tight_layout()
savefig(fig, '06_residual_analysis')
plt.show()

## 6. Statistical Tests on Residuals

In [ ]:
print('=== RESIDUAL DIAGNOSTICS ===')
print(f'{"Model":<12} {"Mean Resid":>12} {"Std Resid":>10} {"Durbin-Watson":>15} {"Shapiro p":>12}')
print('-' * 65)

for model_name, pred in predictions.items():
    residuals = actual - pred.values

    dw = durbin_watson(residuals)

    if len(residuals) >= 3:
        _, shapiro_p = stats.shapiro(residuals)
    else:
        shapiro_p = float('nan')

    print(f'{model_name:<12} {residuals.mean():>12.3f} {residuals.std():>10.3f} '
          f'{dw:>15.3f} {shapiro_p:>12.4f}')

print('\nDurbin-Watson: ~2 = no autocorrelation | <1 or >3 = autocorrelation')
print('Shapiro p > 0.05 = residuals are approximately normal')

## 7. Final Recommendation

In [ ]:
best_model = leaderboard.iloc[0]['Model']
best_rmse  = leaderboard.iloc[0]['RMSE']
best_mape  = leaderboard.iloc[0]['MAPE%']

print('=' * 60)
print(f'  🏆  BEST MODEL: {best_model}')
print(f'       RMSE  = {best_rmse:.3f}')
print(f'       MAPE% = {best_mape:.2f}%')
print('=' * 60)
print()
print('FULL LEADERBOARD:')
print(leaderboard.to_string(index=False))

In [ ]:
# Final radar chart comparison
metrics_for_radar = ['MAE', 'RMSE', 'MAPE%']
model_names = leaderboard['Model'].tolist()

# Normalise metrics (lower = better, so invert for radar)
norm_lb = leaderboard.copy().set_index('Model')
for m in metrics_for_radar:
    max_v = norm_lb[m].max()
    norm_lb[m] = 1 - (norm_lb[m] / max_v)  # 1 = best, 0 = worst

fig = go.Figure()
colors_rd = [PALETTE['primary'], PALETTE['secondary'], PALETTE['success']]

for model_n, color in zip(model_names, colors_rd):
    vals = norm_lb.loc[model_n, metrics_for_radar].tolist()
    vals += vals[:1]  # close the polygon
    theta = metrics_for_radar + [metrics_for_radar[0]]

    fig.add_trace(go.Scatterpolar(
        r=vals, theta=theta,
        name=model_n, fill='toself',
        line=dict(color=color), opacity=0.7
    ))

fig.update_layout(
    title='Model Performance Radar (higher = better)',
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    template=PLOTLY_TEMPLATE,
    showlegend=True, height=480
)
save_plotly(fig, '06_model_radar_comparison')
fig.show()

---
## Final Summary

| Model | MAE | RMSE | MAPE% | Verdict |
|---|---|---|---|---|
| SARIMA | — | — | — | Classical; good for stationary data |
| Prophet | — | — | — | Best for short series with seasonality |
| ETS | — | — | — | Lightweight; robust to outliers |

> 🏆 **Recommended for production:** The model with lowest RMSE on the hold-out set.

### Data Limitations
- Dataset spans **~10 days** — a short window for time series analysis
- A real-world deployment would benefit from 12+ months of daily posting data
- Seasonality patterns would become clearer with longer historical data

### What to Do Next
- Collect rolling monthly data from LinkedIn or job board APIs
- Retrain models monthly with expanding window
- Add external signals: unemployment rate, tech stock indices, GDP data

---
*Analysis complete. All outputs saved to `outputs/figures/` and `outputs/reports/`.*